# Feature Robustness: SPO+ vs MSE vs Weber BLL

**Hypothesis**: SPO+ focuses on decision-relevant features, making it robust to perturbations of irrelevant features. MSE treats all features equally, so its decisions are more sensitive to irrelevant noise.

**Experiment**: Train agents on the same knapsack episodes. At periodic evaluation points, perturb irrelevant vs relevant features and measure how much decisions change.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (14, 5)})

# --- Load results ---
RESULTS_DIR = '../results/ablations/feature_robustness'
DATASET = 'knapsack_32'
N_ROUNDS = 500
N_SEEDS = 3

results_path = f'{RESULTS_DIR}/{DATASET}_n{N_ROUNDS}_s{N_SEEDS}.json'
all_results = None
try:
    with open(results_path) as f:
        all_results = json.load(f)
    print(f'Loaded {results_path}')
    print(f'Seeds: {list(all_results.keys())}')
    seed0 = all_results[list(all_results.keys())[0]]
    print(f'Agents: {list(seed0["agents"].keys())}')
    print(f'Feature relevance: {seed0["feature_relevance"]}')
    print(f'Relevant dims: {seed0["relevant_dims"]}, Irrelevant dims: {seed0["irrelevant_dims"]}')
except FileNotFoundError:
    print(f'File not found: {results_path}')
    print('Run the experiment first:')
    print(f'  python experiments/ablations/feature_robustness.py --dataset {DATASET} --n_rounds {N_ROUNDS} --n_seeds {N_SEEDS}')

File not found: ../results/ablations/feature_robustness/knapsack_32_n500_s3.json
Run the experiment first:
  python experiments/ablations/feature_robustness.py --dataset knapsack_32 --n_rounds 500 --n_seeds 3


In [2]:
def extract_metric_series(all_results, agent_name, metric_key):
    seeds = sorted(all_results.keys())
    rounds = [m['round'] for m in all_results[seeds[0]]['agents'][agent_name]['eval_metrics']]
    per_seed = []
    for s in seeds:
        vals = [m.get(metric_key, np.nan)
                for m in all_results[s]['agents'][agent_name]['eval_metrics']]
        per_seed.append(vals)
    per_seed = np.array(per_seed)
    return np.array(rounds), np.nanmean(per_seed, axis=0), np.nanstd(per_seed, axis=0)

AGENT_COLORS = {'BayesianSPO': '#d62728', 'BayesianMSE': '#1f77b4', 'WeberBLL-Online': '#2ca02c'}
AGENT_LABELS = {'BayesianSPO': 'SPO+', 'BayesianMSE': 'MSE', 'WeberBLL-Online': 'Weber BLL'}

if all_results is not None:
    AGENTS = list(all_results[list(all_results.keys())[0]]['agents'].keys())
else:
    AGENTS = list(AGENT_COLORS.keys())
    print('No results loaded — remaining cells will be skipped.')

No results loaded — remaining cells will be skipped.


## 1. Decision Change: Irrelevant vs Relevant Perturbations Over Time

The core plot. Lower irrelevant decision change = agent focuses on relevant features.

In [3]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    noise_scale = 1.0
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    for ax, agent_name in zip(axes, AGENTS):
        label = AGENT_LABELS.get(agent_name, agent_name)
        color = AGENT_COLORS.get(agent_name, 'gray')
        r, m_irr, s_irr = extract_metric_series(
            all_results, agent_name, f'irrelevant_noise{noise_scale}_decision_change')
        ax.plot(r, m_irr, '-o', color=color, alpha=0.9, label='Irrelevant features', markersize=4)
        ax.fill_between(r, m_irr - s_irr, m_irr + s_irr, alpha=0.15, color=color)
        r, m_rel, s_rel = extract_metric_series(
            all_results, agent_name, f'relevant_noise{noise_scale}_decision_change')
        ax.plot(r, m_rel, '--s', color=color, alpha=0.5, label='Relevant features', markersize=4)
        ax.fill_between(r, m_rel - s_rel, m_rel + s_rel, alpha=0.08, color=color)
        ax.set_title(label, fontsize=14, fontweight='bold')
        ax.set_xlabel('Round')
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('Decision Change Rate\n(frac arms flipped)')
    fig.suptitle(f'Decision Sensitivity to Feature Perturbation (noise σ={noise_scale})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

Skipped — no results loaded.


## 2. Irrelevant/Relevant Ratio Over Time (Focus Metric)

Ratio of decision change under irrelevant vs relevant perturbation. **Lower = more focused on decision-relevant features.** If SPO+ ignores irrelevant features, its ratio should be lower than MSE.

In [4]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    for agent_name in AGENTS:
        label = AGENT_LABELS.get(agent_name, agent_name)
        color = AGENT_COLORS.get(agent_name, 'gray')
        seeds = sorted(all_results.keys())
        rounds = [m['round'] for m in all_results[seeds[0]]['agents'][agent_name]['eval_metrics']]
        ratios_per_seed = []
        for s in seeds:
            metrics_list = all_results[s]['agents'][agent_name]['eval_metrics']
            ratios = []
            for m in metrics_list:
                irr = m.get(f'irrelevant_noise{noise_scale}_decision_change', 0)
                rel = m.get(f'relevant_noise{noise_scale}_decision_change', 1e-8)
                ratios.append(irr / max(rel, 1e-8))
            ratios_per_seed.append(ratios)
        ratios_arr = np.array(ratios_per_seed)
        mean_ratio = ratios_arr.mean(axis=0)
        std_ratio = ratios_arr.std(axis=0)
        ax.plot(rounds, mean_ratio, '-o', color=color, label=label, markersize=5, linewidth=2)
        ax.fill_between(rounds, mean_ratio - std_ratio, mean_ratio + std_ratio, alpha=0.15, color=color)
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='Equal sensitivity')
    ax.set_xlabel('Round')
    ax.set_ylabel('Irrelevant / Relevant\nDecision Change Ratio')
    ax.set_title(f'Feature Focus Ratio (noise σ={noise_scale})\nLower = more decision-focused', fontsize=13)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

Skipped — no results loaded.


## 3. Prediction Change vs Decision Change (Decoupling)

SPO+ might change predictions on irrelevant features but not decisions — the loss teaches it that those prediction changes don't matter. MSE's prediction changes should translate directly to decision changes.

In [5]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for group_name, ax in zip(['irrelevant', 'relevant'], axes):
        for agent_name in AGENTS:
            label = AGENT_LABELS.get(agent_name, agent_name)
            color = AGENT_COLORS.get(agent_name, 'gray')
            r, m_pred, _ = extract_metric_series(
                all_results, agent_name, f'{group_name}_noise{noise_scale}_pred_rmse_change')
            r, m_dec, _ = extract_metric_series(
                all_results, agent_name, f'{group_name}_noise{noise_scale}_decision_change')
            ax.scatter(m_pred, m_dec, c=color, label=label, s=60, alpha=0.7, edgecolors='k', linewidth=0.5)
            ax.plot(m_pred, m_dec, '-', color=color, alpha=0.3, linewidth=1)
        ax.set_xlabel('Prediction RMSE Change')
        ax.set_ylabel('Decision Change Rate')
        ax.set_title(f'{group_name.capitalize()} Feature Perturbation', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
    fig.suptitle('Prediction Sensitivity vs Decision Sensitivity', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

Skipped — no results loaded.


## 4. Test Regret Under Perturbation Across Noise Scales

Bar chart: final test regret at the last evaluation point, for each noise scale and perturbation type.

In [6]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    noise_scales = [0.5, 1.0, 2.0]
    seeds = sorted(all_results.keys())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for group_name, ax in zip(['irrelevant', 'relevant'], axes):
        x = np.arange(len(noise_scales))
        width = 0.25
        for i, agent_name in enumerate(AGENTS):
            label = AGENT_LABELS.get(agent_name, agent_name)
            color = AGENT_COLORS.get(agent_name, 'gray')
            means, stds = [], []
            for ns in noise_scales:
                vals = []
                for s in seeds:
                    last = all_results[s]['agents'][agent_name]['eval_metrics'][-1]
                    vals.append(last.get(f'{group_name}_noise{ns}_test_regret', 0))
                means.append(np.mean(vals))
                stds.append(np.std(vals))
            ax.bar(x + i * width, means, width, yerr=stds, label=label,
                   color=color, alpha=0.8, capsize=3, edgecolor='k', linewidth=0.5)
        ax.set_xticks(x + width)
        ax.set_xticklabels([f'σ={ns}' for ns in noise_scales])
        ax.set_title(f'{group_name.capitalize()} Feature Perturbation', fontsize=13)
        ax.set_xlabel('Noise Scale')
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    axes[0].set_ylabel('Test Regret')
    fig.suptitle('Test Regret Under Feature Perturbation (Final Evaluation)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

Skipped — no results loaded.


## 5. Cumulative Regret Over Time (Standard Comparison)

In [7]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    for agent_name in AGENTS:
        label = AGENT_LABELS.get(agent_name, agent_name)
        color = AGENT_COLORS.get(agent_name, 'gray')
        r, m, s = extract_metric_series(all_results, agent_name, 'cumulative_regret')
        ax.plot(r, m, '-', color=color, label=label, linewidth=2)
        ax.fill_between(r, m - s, m + s, alpha=0.15, color=color)
    ax.set_xlabel('Round')
    ax.set_ylabel('Cumulative Regret')
    ax.set_title('Cumulative Regret (Standard Training)', fontsize=13)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

Skipped — no results loaded.


## 6. Summary Table

In [8]:
if all_results is None:
    print('Skipped — no results loaded.')
else:
    import pandas as pd
    rows = []
    for agent_name in AGENTS:
        label = AGENT_LABELS.get(agent_name, agent_name)
        for ns in [0.5, 1.0, 2.0]:
            irr_dec, rel_dec, irr_reg, rel_reg = [], [], [], []
            for s in seeds:
                last = all_results[s]['agents'][agent_name]['eval_metrics'][-1]
                irr_dec.append(last.get(f'irrelevant_noise{ns}_decision_change', 0))
                rel_dec.append(last.get(f'relevant_noise{ns}_decision_change', 0))
                irr_reg.append(last.get(f'irrelevant_noise{ns}_test_regret', 0))
                rel_reg.append(last.get(f'relevant_noise{ns}_test_regret', 0))
            ratio = np.mean(irr_dec) / max(np.mean(rel_dec), 1e-8)
            rows.append({
                'Agent': label, 'Noise σ': ns,
                'Irr. Dec. Change': f'{np.mean(irr_dec):.4f} ± {np.std(irr_dec):.4f}',
                'Rel. Dec. Change': f'{np.mean(rel_dec):.4f} ± {np.std(rel_dec):.4f}',
                'Focus Ratio': f'{ratio:.3f}',
                'Irr. Test Regret': f'{np.mean(irr_reg):.2f} ± {np.std(irr_reg):.2f}',
                'Rel. Test Regret': f'{np.mean(rel_reg):.2f} ± {np.std(rel_reg):.2f}',
            })
    df = pd.DataFrame(rows)
    df.style.set_caption('Feature Robustness Summary (Final Evaluation Point)')

Skipped — no results loaded.
